# Getting Started with JABS - Live Walkthrough

Welcome to *Getting Started with JABS*, a hands-on introduction to quantifying animal behavior from pose data using the Kumar Lab's JABS pipeline.

**Run the next cell now** - it sets up your environment and takes a minute or two. Keep reading while it runs.

## Learning Goals

By the end of this walkthrough, you should be able to:

- Explain what JABS does, and what makes a behavior **well defined** enough to classify
- Label a behavior in the JABS GUI, using behavior search, zoom and contrast, and the active learning loop
- Explain how cross-validation grouping affects how trustworthy a classifier's score is, and read the cross-validation scores in the JABS training report
- Evaluate a classifier against **dense ground truth**, first frame by frame and then bout by bout after **filter + stitch**
- Judge agreement between annotators, and check that predicted behavior is biologically plausible

### Not Covered in This Workshop

These topics matter, but they are not part of this session. Each has a page on the course site if you want more:

- **Pose estimation** (how keypoints are produced, and how confidence maps work): [Pose Estimation](https://kumarlabjax.github.io/getting-started-with-jabs/02_pose_estimation.html)
- **The pose file format** (HDF5): [Data Ingestion](https://kumarlabjax.github.io/getting-started-with-jabs/01_data_ingestion.html)
- **Pose quality control** (confidence thresholds, failure modes, which sessions to trust): [Precomputed Tracking & QC](https://kumarlabjax.github.io/getting-started-with-jabs/03_precomputed_tracking.html)
- **Annotation and classification in more depth**: [Annotation and Classification with JABS](https://kumarlabjax.github.io/getting-started-with-jabs/04_jabs_classification.html)
- **Bringing your own data**: [Your Own Data](https://kumarlabjax.github.io/getting-started-with-jabs/deep_dive_your_data.html)

Offline copies of these pages are in `docs/` in your clone.

In [ ]:
# Why: one cell that gets the environment ready - find the repo, install the
# pinned dependencies, and confirm each one imported - so every later cell can
# assume its packages exist.
import importlib
import re
import subprocess
import sys
from pathlib import Path

# Find the project root: this notebook lives in student_notebooks/live_demo/.
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    project_root = _nb_file.parent.parent.parent
except NameError:
    current = Path.cwd().resolve()
    for _ in range(10):
        if (current / "requirements.txt").exists() and (current / "student_notebooks").is_dir():
            project_root = current
            break
        current = current.parent
    else:
        raise RuntimeError("Could not find project root.")

print(f"Project root: {project_root}")
print("Installing pinned dependencies (this takes a minute or two)...\n")
# Install into the interpreter this notebook's kernel is running.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(project_root / "requirements.txt")],
    check=True,
)


def parse_requirements(path):
    """Parse a requirements file into package names and minimum versions.

    Args:
        path: Path to the requirements file.

    Returns:
        A list of (pip_name, min_version) tuples. min_version is "0" when a line
        pins no version. Blank lines and comments are skipped.
    """
    entries = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name = re.split(r"[><=!\[]", line)[0].strip()
        ver_match = re.search(r"[><=!]=?\s*([\d.]+)", line)
        min_ver = ver_match.group(1) if ver_match else "0"
        entries.append((name, min_ver))
    return entries


# pip package name -> import name, for packages where they differ
IMPORT_NAME = {
    "opencv-python": "cv2",
    "jabs-behavior-classifier": "jabs",
    "pyDataverse": "pyDataverse",
    "PyYAML": "yaml",
}


def _version_tuple(v):
    """Convert a version string into a comparable tuple.

    Args:
        v: A version string such as "1.24.3" or "2.0rc1".

    Returns:
        A (major, minor, patch) tuple of ints. Missing or non-numeric parts
        become 0.
    """
    # Keep only the digits of each segment (drops suffixes like "rc1") and pad to 3 parts.
    parts = []
    for seg in str(v).split(".")[:3]:
        digits = "".join(c for c in seg if c.isdigit())
        parts.append(int(digits) if digits else 0)
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)


# Check each requirement: import it and compare its version with the minimum.
requirements = parse_requirements(project_root / "requirements.txt")
col1, col2 = 28, 12
print("Package".ljust(col1) + "Installed".ljust(col2) + "Status")
print("-" * 50)
ok_count = 0
for pkg_name, min_ver in requirements:
    import_name = IMPORT_NAME.get(pkg_name, pkg_name)
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "unknown")
        ok = ver == "unknown" or _version_tuple(ver) >= _version_tuple(min_ver)
        status = "[OK]" if ok else "[OUTDATED]"
        ok_count += 1 if ok else 0
    except ImportError:
        ver, status = "-", "[MISSING]"
    print(pkg_name.ljust(col1) + str(ver).ljust(col2) + status)

print()
print(f"{ok_count}/{len(requirements)} packages verified.")
if ok_count < len(requirements):
    print("Install anything missing or outdated with: pip install -r requirements.txt")

## How to Read the Plots

An **ethogram** is a timeline of behavior: time runs left to right, and the color at each moment says what the animal was doing. Many plots below are ethograms.

Every plot in this notebook uses the same colors, so you only learn them once. They match the behavior timeline in the JABS GUI.

- **Orange**: the behavior is present (grooming)
- **Blue**: not the behavior (confirmed not grooming)
- **Light gray**: no label, or not reviewed

An **ethogram** shows one row per video, per annotator, or per processing stage (for example *Labeled*, *Predicted*, *Filter + stitch*). The rows are separate **facets**: when there is more than one video or annotator, we add rows, we never change the colors. The row label always says what you are looking at.

The accuracy, recall, precision and F1 bars also keep one color each wherever they appear; when two stages are compared (raw and filter + stitch), the lighter bar is raw and the solid bar is filter + stitch. The cell below defines these colors and draws the key.

In [ ]:
# Why: one place that defines the colors and the plotting helpers used by every figure
# below, and draws the key for them.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

BEHAVIOR_COLOR = "#FFA500"      # behavior present (orange, as in the JABS GUI)
NOT_BEHAVIOR_COLOR = "#0056E5"  # not the behavior (blue, as in the JABS GUI)
UNREVIEWED_COLOR = "#D9D9D9"    # no label / not reviewed
DISAGREE_COLOR = "#222222"      # frames where two annotators disagree
METRIC_COLORS = {"Accuracy": "#6B6B6B", "Recall": "#009E73", "Precision": "#7B3294", "F1": "#B2182B"}


def _runs(mask):
    """Find the runs of True in a boolean array.

    Args:
        mask: A 1-D boolean array.

    Returns:
        A list of (start, length) pairs, one per run of True, in frame order.
    """
    padded = np.concatenate([[False], np.asarray(mask, dtype=bool), [False]])
    change = np.flatnonzero(padded[1:] != padded[:-1])
    return list(zip(change[::2], change[1::2] - change[::2]))


def ethogram(rows, title=None, width=12, row_height=0.5, xlim=None):
    """Draw an ethogram with one facet (row) per video, annotator or stage.

    Args:
        rows: A list of (label, frames) pairs, or (label, frames, "disagree") for a
            row that marks disagreement. frames is a per-frame array: -1 not reviewed,
            0 not behavior, 1 behavior.
        title: Optional figure title.
        width: Figure width in inches.
        row_height: Height of each row in inches.
        xlim: Optional (first_frame, last_frame) window to zoom into. Frames are then ticked individually.

    Returns:
        The matplotlib figure.
    """
    n_frames = max(len(r[1]) for r in rows)
    fig, axes = plt.subplots(
        len(rows), 1, sharex=True, squeeze=False, figsize=(width, row_height * len(rows) + 0.9)
    )
    for ax, row in zip(axes[:, 0], rows):
        label, frames = row[0], np.asarray(row[1])
        if len(row) > 2 and row[2] == "disagree":
            ax.broken_barh(_runs(frames == 1), (0, 1), facecolors=DISAGREE_COLOR)
        else:
            ax.broken_barh([(0, len(frames))], (0, 1), facecolors=UNREVIEWED_COLOR)
            ax.broken_barh(_runs(frames == 0), (0, 1), facecolors=NOT_BEHAVIOR_COLOR)
            ax.broken_barh(_runs(frames == 1), (0, 1), facecolors=BEHAVIOR_COLOR)
        ax.set_xlim(*(xlim if xlim else (0, n_frames)))
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=9)
        for spine in ax.spines.values():
            spine.set_visible(False)
    if xlim:
        axes[-1, 0].set_xticks(np.arange(xlim[0], xlim[1] + 1, 5))
        for ax in axes[:, 0]:
            ax.grid(axis="x", color="white", linewidth=0.8, alpha=0.9)
    axes[-1, 0].set_xlabel("Frame (30 frames = 1 second)")
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    return fig


def metric_bars(ax, stages, title=None):
    """Draw accuracy, recall, precision and F1 bars, one color per metric.

    Args:
        ax: The matplotlib axes to draw on.
        stages: A dict of stage label to {metric name: value}, in the order they
            should appear. All but the last stage are drawn lighter.
        title: Optional axes title.
    """
    names = list(METRIC_COLORS)
    width = 0.8 / len(stages)
    for k, (stage_label, values) in enumerate(stages.items()):
        alpha = 1.0 if k == len(stages) - 1 else 0.45
        xs = np.arange(len(names)) + (k - (len(stages) - 1) / 2) * width
        bars = ax.bar(xs, [values[n] for n in names], width, color=[METRIC_COLORS[n] for n in names], alpha=alpha)
        for bar, name in zip(bars, names):
            ax.text(bar.get_x() + bar.get_width() / 2, values[name] + 0.02, f"{values[name]:.2f}",
                    ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Score")
    if title:
        ax.set_title(title)
    if len(stages) > 1:
        shades = [("#6B6B6B", 1.0 if k == len(stages) - 1 else 0.45) for k in range(len(stages))]
        ax.legend(handles=[Patch(facecolor=c, alpha=a, label=lab) for (c, a), lab in zip(shades, stages)],
                  loc="lower right", fontsize=8)


# The key: ethogram colors on the left, metric colors on the right.
fig, (ax_e, ax_m) = plt.subplots(1, 2, figsize=(12, 1.4))
ax_e.axis("off")
ax_e.legend(
    handles=[Patch(color=BEHAVIOR_COLOR, label="Behavior present (grooming)"),
             Patch(color=NOT_BEHAVIOR_COLOR, label="Not the behavior"),
             Patch(color=UNREVIEWED_COLOR, label="No label / not reviewed")],
    loc="center left", ncol=1, frameon=False, title="Ethograms", alignment="left",
)
ax_m.axis("off")
ax_m.legend(handles=[Patch(color=c, label=n) for n, c in METRIC_COLORS.items()],
            loc="center left", ncol=2, frameon=False, title="Metrics", alignment="left")
plt.tight_layout()
plt.show()

## What Is JABS?

**JABS** (the JAX Animal Behavior System) is an open-source tool from the Kumar Lab for **labeling and classifying animal behavior from pose data**. Instead of scoring hours of video by hand, you annotate a modest number of example frames for a behavior of interest, and JABS trains a machine-learning classifier to find that behavior across the rest of your data. See the [JABS documentation, User Guide: Overview](https://kumarlabjax.github.io/JABS-behavior-classifier/user-guide/overview/).

The pipeline this walkthrough follows:

```
video  ->  pose estimation  ->  feature extraction  ->  behavior classifier  ->  behavior labels
              (keypoints)          (JABS)                (this course: XGBoost)     (ethograms)
```

Choudhary, A., Geuther, B. Q., Sproule, T. J., Beane, G., Kohar, V., Trapszo, J., & Kumar, V. "JAX Animal Behavior System (JABS), a genetics-informed, end-to-end advanced behavioral phenotyping platform for the laboratory mouse." eLife 107259 (2026). Source: https://github.com/KumarLabJax/JABS-behavior-classifier

## What Makes a Behavior Well Defined?

A classifier can only learn what you can describe. Before labeling anything, write down what the behavior is. A usable definition has a few parts:

- **It is visible in the pose.** JABS sees keypoints, not video. If the behavior does not change where the body parts are, or how they move, there is nothing for the classifier to learn from.
- **It has a clear start and end.** You should be able to say which frame a bout begins and ends on, so two people can place the boundaries in nearly the same place.
- **It has examples and non-examples.** JABS needs both classes: frames where the behavior is present and frames where it is confirmed absent. "Not grooming" is a label you make on purpose, not whatever is left over.
- **People agree on it.** If two annotators watching the same video disagree about what counts, the classifier inherits that disagreement. We measure this later with inter-annotator agreement.

We will show the definition we use for grooming, with example bouts from video, in the slides.

## The Data

This walkthrough uses the **JABS grooming** example project from the Zenodo deposit "JABS Behavior Classifier Example Projects": single-mouse open-field sessions with a pose file and video for each.

- The **practice set** built for you has 10 sessions. About half of them keep their original labels, and the other half have none, so you label those yourself in the JABS GUI. The full Zenodo deposit has 25 videos; to keep a complete pass (features, training and classifying every session) fast in a live session, the practice set keeps only the 10 shortest, and the videos that were removed are listed in its README.
- Session **2010** is held out as the **dense ground-truth** test set: a video where nearly every frame was reviewed. It is never used for training (more on that when we evaluate).
- Every practice session already passed pose **quality control**. We do not cover how that was done here; see [Precomputed Tracking & QC](https://kumarlabjax.github.io/getting-started-with-jabs/03_precomputed_tracking.html). `qc_passed_sessions.json` in the practice set lists them.

**Citation:**

> Geuther, Brian. "JABS Behavior Classifier Example Projects." Zenodo, 2024. https://doi.org/10.5281/zenodo.16697331

**License:** CC BY-NC-4.0

In [ ]:
# Why: import everything the rest of the notebook needs in one place - HDF5
# reading, plotting, the JABS keypoint schema, and a helper that finds the JABS
# command-line tools reliably.
import json
import os
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from jabs.core.abstract.pose_est import PoseEstimation


def jabs_exe(name):
    """Find a JABS console script, preferring the one beside this Python.

    A bare name such as "jabs-cli" only works when the environment's scripts
    folder is on PATH, which is not true when the notebook runs from an editor
    or from a different shell than the launcher.

    Args:
        name: Console script name, for example "jabs-cli".

    Returns:
        The full path to the script if it exists beside the interpreter,
        otherwise the bare name so the OS falls back to a PATH lookup.
    """
    exe_dir = Path(sys.executable).parent
    for candidate in (exe_dir / name, exe_dir / f"{name}.exe",
                      exe_dir / "Scripts" / f"{name}.exe"):
        if candidate.exists():
            return str(candidate)
    return name  # fall back to PATH lookup

# Why: the JABS command-line tools draw progress spinners with Unicode characters; on
# Windows a non-UTF-8 console encoding makes them crash. Child processes inherit this.
os.environ["PYTHONIOENCODING"] = "utf-8"


def clean_output(text):
    """Drop the progress-spinner lines JABS tools print, keeping everything else.

    Args:
        text: Captured stdout or stderr of a JABS command.

    Returns:
        The text without spinner lines (lines with Braille spinner characters) or blank lines.
    """
    keep = [ln for ln in (text or "").splitlines()
            if ln.strip() and not any("\u2800" <= ch <= "\u28ff" for ch in ln)]
    return "\n".join(keep)

In [ ]:
# Why: knobs papermill/CI can override without editing the notebook body.
# parameters
data_subdir = ""   # data/ subfolder to read; "" = auto-detect.
stitch_gap = 30    # postprocessing: stitch predicted bouts separated by a gap of at most this many frames
min_bout = 30      # postprocessing: drop predicted bouts shorter than this many frames (30 frames = 1 second)

In [ ]:
# Why: pick which dataset this notebook reads - the practice set you will label
# in the GUI if it exists, otherwise a fallback, so the notebook still runs.
# Data source - three tiers, first one present wins:
#   Practice set: data/zenodo_scrubbed/jabs_grooming/ - pose + video; about
#                 half the videos keep their original labels, the rest have none
#                 (built by start_jabs.ps1 / start_jabs.sh);
#                 this is the JABS project you will label in the GUI below
#   Full:         data/zenodo_no_gt_cached/jabs_grooming/ - the Zenodo download without the
#                 held-out session 2010, features cached (student_scripts/prepare_training_set.py)
#   Toy:          data/toy/ - small committed sample
_scrubbed = project_root / "data" / "zenodo_scrubbed" / "jabs_grooming"
_zenodo = project_root / "data" / "zenodo_no_gt_cached" / "jabs_grooming"
if data_subdir:
    data_dir = project_root / "data" / data_subdir
elif list(_scrubbed.glob("*_pose_est_v6.h5")):
    data_dir = _scrubbed
elif list(_zenodo.glob("*_pose_est_v6.h5")):
    data_dir = _zenodo
else:
    data_dir = project_root / "data" / "toy"

# Every pose file in the chosen set, in a stable order.
pose_files = sorted(data_dir.glob("*_pose_est_v*.h5"))
print(f"Data directory: {data_dir}")
print(f"Pose files found: {len(pose_files)}")
for f in pose_files[:6]:
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
if len(pose_files) > 6:
    print(f"  ... and {len(pose_files) - 6} more")

# Stop early, with a fix, if nothing was found.
if not pose_files:
    raise FileNotFoundError(
        f"No pose files in {data_dir}.\n"
        "Run the one-click launcher (student_scripts/start_jabs.*), or: "
        "python student_scripts/download_zenodo.py --files jabs_grooming.zip --extract"
    )

## Keypoints and the Skeleton

JABS works from **pose**: for every frame, the position of 12 keypoints on the mouse (nose, ears, base of the tail, and so on). The pose for each session is stored in one HDF5 file (`{session}_pose_est_v6.h5`). This is the file format JABS reads; to learn what is inside it, see [Data Ingestion](https://kumarlabjax.github.io/getting-started-with-jabs/01_data_ingestion.html).

Each keypoint also has a **confidence**: how sure the pose model was about where it put that point. How confidence is produced is covered in [Pose Estimation](https://kumarlabjax.github.io/getting-started-with-jabs/02_pose_estimation.html).

![Mouse keypoint positions](../assets/keypoint_diagram.svg)

A **skeleton** connects anatomically adjacent keypoints, turning the points into a recognizable mouse shape. Here is one well-tracked frame from the first session in the practice set.

In [ ]:
# Why: show the 12-keypoint skeleton on one frame so "keypoint" and "skeleton" are
# concrete. This is the only look at the pose data we take; the features JABS
# computes for classification are built from these points.
# Keypoint names in anatomical order, derived from the JABS package definition.
KEYPOINT_NAMES = [k.name.lower() for k in PoseEstimation.KeypointIndex]

# Connect each neighboring pair in the JABS chain definitions (bones).
SKELETON_EDGES = [
    (KEYPOINT_NAMES[int(s)], KEYPOINT_NAMES[int(e)])
    for chain in PoseEstimation.FULL_CONNECTED_SEGMENTS
    for s, e in zip(chain, chain[1:])
]

pose_path = pose_files[0]
with h5py.File(str(pose_path), "r") as f:
    points = f["poseest/points"][:, 0, :, :]       # (n_frames, 12, 2), stored [y, x]
    confidence = f["poseest/confidence"][:, 0, :]  # (n_frames, 12)

# Use the frame with the highest mean keypoint confidence.
best_frame = int(confidence.mean(axis=1).argmax())
xs, ys = points[best_frame, :, 1], points[best_frame, :, 0]

fig, ax = plt.subplots(figsize=(6, 6))
for start, end in SKELETON_EDGES:
    si, ei = KEYPOINT_NAMES.index(start), KEYPOINT_NAMES.index(end)
    ax.plot([xs[si], xs[ei]], [ys[si], ys[ei]], "k-", alpha=0.4, linewidth=1.5)
for i, kp_name in enumerate(KEYPOINT_NAMES):
    ax.scatter(xs[i], ys[i], s=80, zorder=5)
    ax.annotate(kp_name, (xs[i], ys[i]), fontsize=6, xytext=(4, 4), textcoords="offset points")
ax.set_title(f"Pose skeleton - {pose_path.name}, frame {best_frame}")
ax.set_aspect("equal")
ax.invert_yaxis()  # image coordinates: y=0 is the top of the frame
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
plt.tight_layout()
plt.show()

## Windows: Behavior Is Temporal

A single frame rarely says what an animal is doing: a still mouse could be resting or freezing, and a paw at the face could be the start or the end of a grooming bout. Behavior unfolds over time. So JABS does not compute features from one frame alone. For every frame it also computes features over a **window**: the stretch of frames before and after it in which features are calculated, such as the average speed or how much an angle changes across the window.

Window size is a scientific choice. A short window captures quick movements and a long one captures slow, repeated movements. The grooming classifier in this course uses a window size of 60 (frames before and after the current frame), a project setting.

The idea of window features comes from Kristin Branson and the JAABA system ([Kabra et al. 2013](https://www.nature.com/articles/nmeth.2281); [code](https://github.com/kristinbranson/JAABA)). Credit for it goes to her.

## Getting Your Data into JABS

Little of this session is about getting data in, so here is the short version.

- A JABS project is a folder with a **pose file and a video for each session**; JABS keeps its own files in a `jabs/` subfolder (labels in `jabs/annotations/`).
- The pose here comes from the Kumar Lab's tracking pipeline. Pose from other tools (SLEAP, DeepLabCut, and others) has to be converted to the JABS pose format first.
- JABS can **export** pose to NWB (`jabs-cli convert-to-nwb`). **NWB import is coming soon**: it is not working yet, and it is being built on top of a redesigned JABS pose file format.
- The principles in this walkthrough (define the behavior, label examples, train, validate against independent ground truth) are **not specific to JABS**. They apply to any pose-based behavior classifier.

For more, see [Data Ingestion](https://kumarlabjax.github.io/getting-started-with-jabs/01_data_ingestion.html), [Your Own Data](https://kumarlabjax.github.io/getting-started-with-jabs/deep_dive_your_data.html), and the [JABS documentation](https://kumarlabjax.github.io/JABS-behavior-classifier/).

## The JABS GUI

The JABS desktop app is where you label. Open the practice set as a project (`data/zenodo_scrubbed/jabs_grooming/`); the one-click launcher builds it for you, or rebuild it by hand with `python student_scripts/make_scrubbed_practice_set.py`.

The [JABS GUI walkthrough](https://kumarlabjax.github.io/getting-started-with-jabs/live_demo_jabs_gui_walkthrough.html) (offline copy: `docs/live_demo_jabs_gui_walkthrough.html` in your clone) shows what each panel does. See also the JABS documentation, User Guide: [GUI](https://kumarlabjax.github.io/JABS-behavior-classifier/user-guide/gui/) and [Labeling](https://kumarlabjax.github.io/JABS-behavior-classifier/user-guide/labeling/). We also hand out a cheatsheet of shortcuts and views.

## JABS Features That Help You Label and Check

**Behavior search** (*Search Behaviors* in the Tools menu, or Ctrl+F, Cmd+F on a Mac) jumps you to frames that match a query. You can search by **label** (where you or someone else labeled the behavior), by **prediction** (frames the classifier is confident or unsure about, which is how you find the frames worth labeling next), or by **timeline annotation** tag. To find the frames the classifier is least sure about, search by **prediction** and choose **low confidence**: those are the best frames to label next.

**Zoom and contrast.** You can zoom into a region of the video frame and reset the view with the restore button, and the video overlay has **brightness** and **contrast** controls. Both help when the behavior is small or the lighting is poor.

**Cross-validation grouping.** In the settings dialog you choose what a *group* is for cross-validation:

- **Individual Animal**: each animal is a group.
- **Video**: each video is a group.
- **Filename Pattern**: a regular expression picks a group name out of each filename, so you can group by anything encoded in it, such as a cage ID.

**When does Train turn on?** It depends on the number of cross-validation iterations (the **Cross Validation k** slider) and on the grouping. With k set to **0** (no cross-validation), Train turns on once the project has at least **20 labeled frames of the behavior and 20 of not-behavior**. With k above 0, it needs enough *groups* that each have 20 or more frames of both classes: at least k groups, and never fewer than 2.

### Classifier Choice and Cross-Validation: What the GUI and CLI Assume

**Classifier choice.** JABS offers Random Forest (its default in the GUI), CatBoost and XGBoost ([JABS docs, Classifier Types](https://kumarlabjax.github.io/JABS-behavior-classifier/user-guide/classifier-types/)). None is a deep neural network. This course uses **XGBoost**, which is also what `jabs-cli export-training` writes unless you pass `--classifier`. Tree-based models suit the distances, angles and speeds JABS computes from keypoints, train quickly, and need far less labeled data than deep models. The choice is stored in the training file, so `jabs-classify` uses whichever algorithm the file names.

**Cross-validation in the GUI.** Clicking **Train** also runs leave-one-group-out cross-validation: whole videos or whole animals are held out, not random frames, so the classifier is tested on data it has never seen. The **Cross Validation k** slider sets how many held-out iterations run, and **All k-fold Cross Validation** runs every fold. The slider starts at **1**, so by default the report reflects a single held-out group, and one unlucky fold can show precision, recall and F1 of 0 for the behavior even when the final classifier, trained on all your labels, works well.

**The CLI assumes something different.** `jabs-classify train` does not cross-validate at all: it fits one classifier on everything in the training file and reports no score. Cross-validation is a separate command, `jabs-cli cross-validation`, which runs **all** splits by default (`-k 0`; note that a slider value of 0 in the GUI gives no cross-validation report instead), lets you choose the grouping (`--grouping-strategy video`, `individual` or `filename`; otherwise the project setting), and can write a Markdown or JSON report.

So the same project can give a noisy one-fold score in the GUI, a full cross-validation from the CLI, or no score from `jabs-classify train`. Check which one you are looking at, raise k (or tick All k-fold) in the GUI to see the spread across groups, and where you can, confirm with a dense, independently reviewed session, as we do below.

## Break: Label Grooming in the JABS GUI

**Step away from this notebook for a few minutes.** The labels the classifier learns from come from you, made in the JABS desktop app. About half of the practice videos already carry the original labels; the rest are yours to label.

1. Open `data/zenodo_scrubbed/jabs_grooming/` as a project in the JABS GUI.
2. Pick a video that has no labels and label **grooming** and **not grooming** frames. Aim for a handful of clear bouts of each. Label the confident core of each bout and avoid its edges: the first and last frames are where people disagree most, so they make the noisiest labels.
3. Train, then use the classifier's predictions to find where it is wrong, and label those frames. This is the **active learning** loop: *label, train, look at the predictions, label what the classifier got wrong, train again*. Each round spends your labeling time on the frames that teach the classifier the most.
4. Save, **Train** one last time and **Classify**, so your classifier (`jabs/classifiers/`) and its predictions on your videos (`jabs/predictions/`) are saved in the project. Then come back and run the next cells: the notebook loads that classifier, it does not train its own.

Think-pair-share: with a neighbor, compare where each of you put the start and end of a grooming bout. You will each label different frames, so your classifiers will differ. We will put numbers on that agreement shortly, and later look at where your classifiers agree.

## Back from the Break: From Your Labels to a Classifier

Load the labels you just made, check them against the pose, then pick up the classifier you trained in the GUI and test it.

In [ ]:
# Why: the behavior to read from the annotation files; change it to analyze
# another behavior in the same project.
# parameters
BEHAVIOR = "grooming"

### Load Your Annotations

JABS saves each session's labels as `unfragmented_labels`: **present** (grooming) and **not present** (confirmed non-grooming) bouts. These include the original labels on the pre-labeled videos and anything you added. We turn them into a bout table and a per-frame array (`1` grooming, `0` not grooming, `-1` unreviewed). The coverage table shows how much of each session was reviewed, since unreviewed frames cannot be scored.

In [ ]:
# Why: read the labels you made in the GUI into two forms - a table of grooming
# bouts (for plotting) and a per-frame label array (for checks and scoring).
ann_dir = data_dir / "jabs" / "annotations"
session_ids = [p.stem.split("_pose_est")[0] for p in pose_files]

bouts = []
frame_labels = {}

for pose_path in pose_files:
    sid = pose_path.stem.split("_pose_est")[0]
    ann_path = ann_dir / f"{sid}.json"

    with h5py.File(str(pose_path), "r") as f:
        n_fr = f["poseest/points"].shape[0]

    # -1 = never reviewed, 0 = reviewed and not grooming, 1 = grooming.
    labels = np.full(n_fr, -1, dtype=np.int8)

    # No annotation file just means no frames were reviewed for this session.
    if ann_path.exists():
        d = json.loads(ann_path.read_text(encoding="utf-8"))
        for identity, behaviors in d.get("unfragmented_labels", {}).items():
            for behavior, intervals in behaviors.items():
                if behavior.lower() != BEHAVIOR:
                    continue
                # Fill the frame array from each labeled interval; only grooming
                # intervals also go in the bout table.
                for iv in intervals:
                    start, end = int(iv["start"]), int(iv["end"])
                    val = 1 if iv.get("present", False) else 0
                    labels[start:end] = val
                    if iv.get("present", False):
                        bouts.append({
                            "session": sid,
                            "start": start,
                            "end": end,
                            "length": end - start,
                        })

    frame_labels[sid] = labels

bouts_df = pd.DataFrame(bouts) if bouts else pd.DataFrame(
    columns=["session", "start", "end", "length"]
)
print(f"Loaded {len(session_ids)} sessions, {len(bouts_df)} grooming bouts total")

In [ ]:
# Why: show how much of each session was reviewed - frames nobody reviewed
# cannot be scored.
rows = []
for pose_path in pose_files:
    sid = pose_path.stem.split("_pose_est")[0]
    with h5py.File(str(pose_path), "r") as f:
        n_fr = f["poseest/points"].shape[0]
    reviewed = int((frame_labels[sid] >= 0).sum())
    grooming = int((frame_labels[sid] == 1).sum())
    # "dense" means every frame in the session was reviewed.
    rows.append({
        "session":         sid,
        "session_frames":  n_fr,
        "reviewed":        reviewed,
        "coverage_pct":    100 * reviewed / n_fr,
        "grooming_frames": grooming,
        "grooming_pct":    100 * grooming / max(reviewed, 1),
        "dense":           reviewed >= n_fr,
    })

coverage_df = pd.DataFrame(rows)
print(coverage_df.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

dense_sessions = coverage_df.loc[coverage_df["dense"], "session"].tolist()
print(f"\nFully reviewed sessions in this project: {dense_sessions}")

In [ ]:
# Why: see the labels in the project as an ethogram - one row per session, orange
# where you (or the original annotator) marked grooming, blue where not grooming,
# gray where nothing was reviewed.
label_rows = [(sid, frame_labels[sid]) for sid in sorted(frame_labels) if (frame_labels[sid] >= 0).any()]
if not label_rows:
    print("No labels found yet - label some sessions in the JABS GUI first, then re-run the loading cells above.")
else:
    ethogram(label_rows, title="Labels in the practice project")
    plt.show()

## Check In

In your own words: what did you do to decide where a grooming bout started and ended, and how much would a second person's answer differ?

### Your Classifier

You trained in the JABS GUI, so the classifier already exists: JABS saved it in the project's `jabs/classifiers/` folder when you clicked **Train**. The notebook does not train. It loads that classifier and uses `jabs-classify classify` to run it on held-out session 2010, the one session it has never seen, and shows you how it did. The practice sessions are training data, so we do not classify them again: for the checks that need predictions on them we read the ones the GUI saved. The command is deterministic: the same classifier on the same pose file always gives the same predictions. What is *not* reproducible is the human step upstream, because two people labeling "grooming" from scratch may not agree. You can run someone else's classifier on your own pose files, but validate it on your own ground truth first, as we do next.

If you have no classifier of your own (you skipped the break, or you are running this notebook unattended), the next cell falls back to a classifier trained ahead of time on every labeled session of the full Zenodo dataset except session 2010. It says which one it is using, and the training report that came with it (the cross-validation table) is shown below.

In [ ]:
# Why: pick the classifier the rest of the notebook evaluates - the one you trained in the
# GUI if there is one, otherwise the shipped fallback - and make sure session 2010 stays
# out of the project, so the evaluation below is a fair test.
import datetime

if (data_dir / "2010_pose_est_v6.h5").exists():
    raise RuntimeError(
        "Session 2010 is in this project, but it must stay held out of training.\n"
        "Rebuild the practice set: python student_scripts/make_scrubbed_practice_set.py"
    )

# The GUI saves each classifier it trains in jabs/classifiers/. With active learning you train
# several times, so always take the newest one.
classifier_dir = data_dir / "jabs" / "classifiers"
saved_classifiers = sorted([*classifier_dir.glob("*.pickle"), *classifier_dir.glob("*.pkl")],
                           key=lambda f: f.stat().st_mtime) if classifier_dir.exists() else []
gui_classifier = saved_classifiers[-1] if saved_classifiers else classifier_dir / f"{BEHAVIOR}.pickle"
fallback_classifier = project_root / "data" / "pretrained" / f"{BEHAVIOR}_zenodo_full.pickle"
fallback_report = project_root / "data" / "pretrained" / f"{BEHAVIOR}_zenodo_full_training_report.md"
live_preds_dir = data_dir / "jabs" / "predictions_live" / BEHAVIOR
live_preds_dir.mkdir(parents=True, exist_ok=True)
# Predictions from an older run or an older practice set would be mistaken for this classifier's.
for old_pred in live_preds_dir.glob("*.h5"):
    old_pred.unlink()

if gui_classifier.exists():
    classifier_out = gui_classifier
    saved = datetime.datetime.fromtimestamp(gui_classifier.stat().st_mtime)
    print(f"Using the classifier you trained in the GUI: {gui_classifier.name} (saved {saved:%Y-%m-%d %H:%M})")
    newest_label = max((f.stat().st_mtime for f in (data_dir / "jabs" / "annotations").glob("*.json")), default=0)
    if newest_label > gui_classifier.stat().st_mtime:
        print("Note: you changed labels after training. Click Train in the GUI again to include them.")
elif fallback_classifier.exists():
    classifier_out = fallback_classifier
    print("No classifier found from the GUI (click Train in the JABS GUI to make one).")
    print(f"Using the shipped fallback: {fallback_classifier.name} (trained on the full Zenodo set, without session 2010).")
else:
    classifier_out = gui_classifier
    print("[MISSING] No classifier from the GUI and no shipped fallback.")
    print(f"Train one in the JABS GUI, or build the fallback: python developer/scripts/make_pretrained_classifier.py")

In [ ]:
# Why: show the cross-validation result JABS wrote when this classifier was trained - the
# training report - so the spread across held-out videos is in front of us, not hidden in a mean.
# Your own report is in the project's jabs/training_logs/; the fallback classifier ships with one.
from IPython.display import Markdown, display

if classifier_out == gui_classifier:
    reports = sorted((data_dir / "jabs" / "training_logs").glob(f"{BEHAVIOR}_*_training_report.md"),
                     key=lambda f: f.stat().st_mtime)
    training_report = reports[-1] if reports else None
else:
    training_report = fallback_report if fallback_report.exists() else None

if training_report is None:
    print("No training report found. Click Train in the JABS GUI to make one; it is saved in jabs/training_logs/.")
else:
    report_text = training_report.read_text(encoding="utf-8")
    # Show the summary and the cross-validation section; skip the long feature-importance table.
    start = report_text.index("## Training Summary") if "## Training Summary" in report_text else 0
    stop = report_text.index("## Feature Importance") if "## Feature Importance" in report_text else len(report_text)
    print(f"Training report: {training_report.name}")
    display(Markdown(report_text[start:stop]))

In [ ]:
# Why: get the predictions the evaluation and the biological checks below need. Held-out
# session 2010 is the only session we classify live: it is a test set the classifier never saw.
# The practice sessions are training data, so we do not classify them again here; we read the
# predictions the JABS GUI saved for them (in your project, else in the training pool the
# fallback classifier was trained in).
# The dense ground-truth session (2010) is held out separately, not in data_dir.
dense_gt_dir = project_root / "data" / "dense_gt_a1"
dense_pose_path = dense_gt_dir / "2010_pose_est_v6.h5"
pool_dir = project_root / "data" / "zenodo_no_gt_cached" / "jabs_grooming"


def has_predictions(path):
    """True if a prediction file holds this behavior's predicted classes."""
    with h5py.File(str(path), "r") as f:
        return f"predictions/{BEHAVIOR}/predicted_class" in f


live_pred_files = []

# 1. Session 2010, classified now with the classifier chosen above.
if not classifier_out.exists():
    print(f"[MISSING] Classifier not found: {classifier_out}")
    print("Skipping the classify step for this run.")
else:
    print(f"Classifying held-out session 2010 with {classifier_out.name} ...")
    result = subprocess.run(
        [jabs_exe("jabs-classify"), "classify",
         "--classifier", str(classifier_out),
         "--input-pose", str(dense_pose_path),
         "--feature-dir", str(dense_gt_dir / "jabs" / "features"),
         "--out-dir", str(live_preds_dir)],
        capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    print("  OK" if result.returncode == 0 else "  FAILED")
    if result.returncode != 0:
        print(clean_output(result.stderr))
    live_pred_files += sorted(live_preds_dir.rglob("*.h5"))

# 2. Practice sessions: predictions the GUI already saved (jabs/predictions/<session>.h5).
qc_path = data_dir / "qc_passed_sessions.json"
if qc_path.exists():
    practice_paths = [data_dir / s["file"] for s in json.loads(qc_path.read_text(encoding="utf-8"))["passed_sessions"]]
else:
    practice_paths = list(pose_files)
found_from = {}
for pose_path in practice_paths:
    stem = pose_path.name.removesuffix("_pose_est_v6.h5")
    for label, src_dir in (("your GUI project", data_dir), ("the training pool", pool_dir)):
        candidate = src_dir / "jabs" / "predictions" / f"{stem}.h5"
        if candidate.exists() and has_predictions(candidate):
            live_pred_files.append(candidate)
            found_from.setdefault(label, []).append(stem)
            break
for label, stems in found_from.items():
    print(f"Predictions for {len(stems)} practice session(s) from {label}: {', '.join(stems)}")
if not found_from:
    print("No GUI predictions found for the practice sessions (click Train and Classify in the JABS GUI).")
    print("The checks below that compare sessions will use session 2010 only.")

print(f"\nPrediction files in use: {len(live_pred_files)}")

# Read each prediction file's per-frame predicted class, keyed by session.
frame_preds = {}
for pf in live_pred_files:
    with h5py.File(str(pf), "r") as f:
        key = f"predictions/{BEHAVIOR}/predicted_class"
        if key in f:
            sid = next((s for s in session_ids + ["2010"] if s in pf.stem), pf.stem)
            frame_preds[sid] = f[key][0].astype(np.int8)

# Quick look: predicted vs annotated grooming frames for the first few sessions.
for sid, arr in list(frame_preds.items())[:5]:
    n_grm = int(arr.sum())
    ann_grm = int((frame_labels.get(sid, np.array([])) == 1).sum())
    print(f"  {sid}: predicted {n_grm} grooming frames  |  annotated {ann_grm}")

## Cross-Validation: Leave One Video Out

Before testing on a video the classifier has never seen, JABS can estimate how well it will do from the labels you already have. **Leave-one-out cross-validation (LOOCV)** does this by rotating which video is held out: hold one video out, train on all the others, score the held-out video, and repeat until every video has had a turn. JABS holds out whole videos (the grouping set in the settings; see *Cross-validation grouping* above), never random frames from videos it trained on, so each score is a test on unseen footage.

You already have this result. When you clicked **Train** in the GUI, JABS ran cross-validation with your settings and wrote a **training report** to the project's `jabs/training_logs/` folder: a Markdown file with one row per held-out group (accuracy, precision, recall, F1), plus the mean. We read it together instead of recomputing it here. From the command line, `jabs-cli cross-validation` produces the same thing: `-k 0` holds every group out once, and `--grouping-strategy video` makes each video a group. Only videos with at least 20 labeled frames of both classes can be held out.

Two things to read in the report:

- **The spread.** Each held-out video gives its own score. When the labels come from only a few videos the scores can differ a lot, because a video with unusual behavior or very few bouts is hard to predict from the others. A single fold is a noisy estimate; the mean over folds is better.
- **What it is not.** Cross-validation reuses the same labeled videos, and the same annotator, for training and testing. It estimates performance on *similar* data. The test on session 2010 below is the independent check.

### What Should You Leave Out? Video, Animal or Arena

JABS lets you choose the unit that is held out in each fold, and the choice decides what the score tells you. The stricter the grouping, the better the score reflects a genuinely new situation, and the harder and noisier it gets.

| Held out | JABS setting | The score estimates performance on... | Watch out for |
|----------|--------------|---------------------------------------|---------------|
| **One video** | Video | a new recording | If the same animal or arena appears in several videos, the other videos have already taught the classifier that animal or arena, so the score is optimistic for a new animal or arena. |
| **One animal** | Individual Animal | a new animal | JABS counts each animal *within a video*. With one mouse per video this is the same as holding out a video (the folds would be identical). To hold out every video of the same animal, put an animal ID in the filenames and use Filename Pattern. |
| **One arena** (or cage, rig, site) | Filename Pattern: a regular expression that picks the arena ID out of each filename | a new physical setup: different camera, lighting or arena | The arena ID has to be in the filenames. This is the strictest test, so scores are usually lower and noisier. |

**Choose the grouping that matches how you will use the classifier.** New videos of the same animals in the same arena: video is enough. A new cohort of animals: hold out animals. A different lab, rig or arena: hold out arenas.

Every grouping needs at least two groups (and at least *k* of them), each with 20 or more labeled frames of both classes, which is why strict groupings are hard to run with few labeled videos.

In this practice set each video has one mouse and the filenames are only session numbers, with no animal or arena ID, so we cannot tell from the filenames whether videos share an animal or an arena. Leave-one-video-out is the cross-validation you will see in your training report, and holding out an animal or an arena across videos would need that ID in the filename.

## Performance: Dense Ground Truth at the Frame Level

Session **2010** is fully reviewed and was held out of training entirely (`data/dense_gt_a1/`), so it is an unbiased test set. We run `jabs-cli evaluate`, which classifies the held-out project with your classifier and compares the predictions against its labels.

**A caution about how this is set up.** Best practice is to have more than one densely labeled video, ideally labeled by more than one person. Here the instructors labeled a single video, to show the idea. One video is not enough to trust a score in real work.

First, the frame-by-frame view: for each frame, did the prediction match the label?

In [ ]:
# Why: score the classifier on the held-out dense ground-truth project with JABS's own
# evaluate command, once, with and without the postprocessing defined above, so the
# frame-level and bout-level views come from the same run.
import os

eval_dir = data_dir / "jabs" / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)
for old_file in eval_dir.glob("*"):
    old_file.unlink()

# Postprocessing pipeline: stitch short gaps first, then drop short bouts.
pp_config = [
    {"stage_name": "BoutStitchingStage", "parameters": {"max_stitch_gap": int(stitch_gap)}},
    {"stage_name": "BoutDurationFilterStage", "parameters": {"min_duration": int(min_bout)}},
]
pp_config_path = eval_dir / "postprocessing.json"
pp_config_path.write_text(json.dumps(pp_config, indent=2), encoding="utf-8")

eval_json = None
if not classifier_out.exists():
    print("[MISSING] No classifier - run the \"Your Classifier\" cells above first.")
else:
    result = subprocess.run(
        [jabs_exe("jabs-cli"), "evaluate", str(dense_gt_dir),
         "--classifier", str(classifier_out), "--behavior", BEHAVIOR,
         "--postprocess-config", str(pp_config_path), "--out-dir", str(eval_dir)],
        capture_output=True, text=True, encoding="utf-8", errors="replace",
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    summary_files = sorted(eval_dir.glob("*_evaluation.json"))
    if result.returncode != 0 or not summary_files:
        print("Evaluation failed:")
        print(clean_output(result.stderr or result.stdout))
    else:
        eval_json = json.loads(summary_files[-1].read_text(encoding="utf-8"))
        print(f"Evaluated {dense_gt_dir.name} with {classifier_out.name}")

In [ ]:
# Why: load session 2010's real labels and our predictions on it - the held-out
# test the classifier never saw.
dense_ann_path = dense_gt_dir / "jabs" / "annotations" / "2010.json"

with h5py.File(str(dense_pose_path), "r") as f:
    n_dense_frames = f["poseest/points"].shape[0]

# Same encoding as before: -1 unreviewed, 0 not grooming, 1 grooming.
dense_labels = np.full(n_dense_frames, -1, dtype=np.int8)
d = json.loads(dense_ann_path.read_text(encoding="utf-8"))
for identity, behaviors in d.get("unfragmented_labels", {}).items():
    for behavior, intervals in behaviors.items():
        if behavior.lower() != BEHAVIOR:
            continue
        for iv in intervals:
            start, end = int(iv["start"]), int(iv["end"])
            dense_labels[start:end] = 1 if iv.get("present", False) else 0

print(f"Dense GT session 2010: {n_dense_frames} frames, "
      f"{int((dense_labels >= 0).sum())} reviewed, {int((dense_labels == 1).sum())} grooming")

# Use this notebook's own live classify output for session 2010 (produced above),
# not a pre-baked file. Session 2010 is not in the practice project, so it was
# never part of training: this is a genuine held-out evaluation.
dense_preds = frame_preds.get("2010")
if dense_preds is None:
    print("[MISSING] No live predictions found for session 2010 - "
          "check that the classify step above ran successfully.")
else:
    print(f"Loaded live predictions for session 2010 ({int(dense_preds.sum())} predicted grooming frames)")

In [ ]:
# Why: look at the data before computing numbers - the labeled and the predicted
# ethogram for session 2010, with behavior and not-behavior both shown.
if dense_preds is None:
    print("No predictions loaded - skipping visual check.")
else:
    n = min(len(dense_labels), len(dense_preds))
    ethogram(
        [("Labeled (a1)", dense_labels[:n]), ("Predicted (raw)", dense_preds[:n])],
        title="Session 2010 (dense ground truth): labels vs. raw predictions",
    )
    plt.show()

### Metrics

Accuracy alone can mislead. A **null classifier** that never predicts grooming scores a high accuracy whenever the behavior is rare, so a high accuracy says little by itself. (Session 2010 is an unusual case: grooming is about half of its reviewed frames, so the null baseline here is low. Most behaviors are rarer.) Read recall, precision and F1 instead; **F1** is the standard summary for imbalanced behavioral data. Two more numbers do not depend on the classifier's decision threshold: **AUROC**, and **5% FPR at 0.nn TPR** (read it as: when only 5% of not-grooming frames may be wrongly called grooming, the classifier catches 0.nn of the grooming frames).

| Metric | Formula | Asks |
|--------|---------|------|
| Accuracy | (TP + TN) / all | What fraction of all frames did we classify correctly? |
| Recall | TP / (TP + FN) | Of all grooming frames, what fraction did we catch? |
| Precision | TP / (TP + FP) | Of frames we called grooming, what fraction actually were? |
| F1 | 2 * P * R / (P + R) | Harmonic mean of precision and recall |

In [ ]:
# Why: show what a classifier that never predicts grooming would score, then the
# real frame-level numbers from JABS's evaluate, so the accuracy has a baseline to beat.

def frame_metrics_table(stage):
    """Return the frame-level metrics `jabs-cli evaluate` reported for one stage.

    Args:
        stage: "raw" or "postprocessed", a key of the evaluate summary's "stages".

    Returns:
        A one-row DataFrame with the confusion counts and the accuracy, precision,
        recall and F1 exactly as the JABS command reported them.
    """
    fr = eval_json["stages"][stage]["overall"]["frames"]
    return pd.DataFrame([{
        "TP": fr["true_positive"], "FP": fr["false_positive"],
        "TN": fr["true_negative"], "FN": fr["false_negative"],
        "Accuracy": fr["accuracy"], "Recall": fr["recall_behavior"],
        "Precision": fr["precision_behavior"], "F1": fr["f1_behavior"],
    }], index=[stage])

if eval_json is None:
    print("No evaluation results - run the cell above first.")
else:
    raw = frame_metrics_table("raw")
    n_total = int(raw[["TP", "FP", "TN", "FN"]].sum(axis=1).iloc[0])
    prevalence = float((raw["TP"] + raw["FN"]).iloc[0]) / n_total
    # The baseline is plain arithmetic on the CLI's counts: the share of reviewed frames that are not grooming.
    print(f"Grooming prevalence in dense GT : {100 * prevalence:.1f}% of reviewed frames")
    print(f"Null model accuracy (never predict grooming): {100 * (1 - prevalence):.1f}%")
    print(f"Classifier accuracy (from jabs-cli evaluate): {100 * float(raw['Accuracy'].iloc[0]):.1f}%\n")
    print("Frame-level, raw predictions:")
    print(raw.round(3).to_string())

    fig, ax = plt.subplots(figsize=(6, 3.5))
    metric_bars(ax, {"Raw": {n: float(raw[n].iloc[0]) for n in METRIC_COLORS}},
                title="Frame-level metrics - dense ground-truth session")
    plt.tight_layout()
    plt.show()

### Reading the ROC Curve

The classifier gives every frame a **probability** of grooming. A **call** of "grooming" means that probability is above a cutoff, 0.5 by default. The ROC curve below moves the cutoff from strict to loose and plots, at each setting, the share of real grooming frames caught (true positive rate, up) against the share of not-grooming frames wrongly called grooming (false positive rate, right). A curve that hugs the top-left corner is better, and the dashed diagonal is guessing. **AUROC** is the area under the curve. The orange dot reads the curve at 5% on the horizontal axis, which gives the "5% FPR at 0.nn TPR" figure.

In [ ]:
# Why: add the two threshold-free numbers - AUROC and 5% FPR at 0.nn TPR - which need the
# classifier's probabilities, not just its yes/no calls.
from sklearn.metrics import roc_auc_score, roc_curve

if dense_preds is None:
    print("No predictions loaded - skipping.")
else:
    pred_file_2010 = next(pf for pf in live_pred_files if "2010" in pf.stem)
    with h5py.File(str(pred_file_2010), "r") as f:
        pred_prob = f[f"predictions/{BEHAVIOR}/probabilities"][0]
    # JABS stores the probability of the *predicted* class; turn it into P(grooming).
    p_grooming = np.where(dense_preds == 1, pred_prob, 1 - pred_prob)

    reviewed = dense_labels >= 0
    y_ref = dense_labels[reviewed]
    scores = p_grooming[:len(dense_labels)][reviewed]
    auroc = roc_auc_score(y_ref, scores)
    fpr, tpr, _ = roc_curve(y_ref, scores)
    tpr_at_5 = float(np.interp(0.05, fpr, tpr))
    print(f"AUROC: {auroc:.3f}")
    print(f"5% FPR at {tpr_at_5:.2f} TPR")

    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.plot(fpr, tpr, color="#333333")
    ax.plot([0, 1], [0, 1], color="#95a5a6", linestyle="--", label="chance")
    ax.scatter([0.05], [tpr_at_5], color=METRIC_COLORS["Recall"], zorder=5, label=f"5% FPR at {tpr_at_5:.2f} TPR")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC curve - session 2010 (AUROC {auroc:.2f})")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

## From Frames to Bouts: Filter and Stitch

Behavior is rarely analyzed one frame at a time. A scientist asks *how many grooming bouts, and how long are they*. A classifier that flickers on and off for a few frames inside one real bout produces many short predicted bouts, even when most frames are right.

JABS **postprocessing** turns frame-level predictions into cleaner bouts with two stages, applied in this order:

- **Stitch** (`BoutStitchingStage`): join two predicted bouts separated by a short gap, up to `max_stitch_gap` frames.
- **Filter** (`BoutDurationFilterStage`): remove predicted bouts shorter than `min_duration` frames.

We use a stitch gap of `stitch_gap` frames and a minimum bout of `min_bout` frames (set in the parameters cell; 30 frames is 1 second). They are scientific choices: choose them from how long your behavior's real bouts and real pauses are.

From here on, everything is evaluated at the **bout level, after filter and stitch**.

Bouts are matched to labeled bouts in two ways: **overlap** (did the classifier notice the bout at all?) and **IoU** (intersection-over-union of the two frame ranges, which also penalizes predicted bouts much longer or shorter than the real one).

In [ ]:
# Why: see what filter + stitch did to the predicted bouts on session 2010 - the labeled
# bouts, the raw predictions and the cleaned predictions on the same time axis - before
# looking at the numbers. The cleaning is done by the JABS command `jabs-cli postprocess`.
def postprocess_with_cli(pred_file, out_file):
    """Run `jabs-cli postprocess` on one prediction file and return the cleaned classes.

    Args:
        pred_file: A prediction HDF5 file written by `jabs-classify classify`.
        out_file: Where to write the postprocessed copy (the input is left unchanged).

    Returns:
        The per-frame predicted classes after filter + stitch, or None if the command failed.
    """
    result = subprocess.run(
        [jabs_exe("jabs-cli"), "postprocess", "--config", str(pp_config_path),
         "--behavior", BEHAVIOR, "--output", str(out_file), str(pred_file)],
        capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    if result.returncode != 0:
        print(clean_output(result.stderr or result.stdout))
        return None
    with h5py.File(str(out_file), "r") as f:
        return f[f"predictions/{BEHAVIOR}/predicted_class_postprocessed"][0]


if dense_preds is None:
    print("No predictions loaded - skipping.")
else:
    pred_file = next(pf for pf in live_pred_files if "2010" in pf.stem)
    dense_preds_pp = postprocess_with_cli(pred_file, eval_dir / "2010_postprocessed.h5")
    n = min(len(dense_labels), len(dense_preds))
    ethogram(
        [
            (f"Labeled (a1)\n{len(_runs(dense_labels[:n] == 1))} bouts", dense_labels[:n]),
            (f"Predicted, raw\n{len(_runs(dense_preds[:n] == 1))} bouts", dense_preds[:n]),
            (f"Predicted, filter + stitch\n{len(_runs(dense_preds_pp[:n] == 1))} bouts", dense_preds_pp[:n]),
        ],
        title="Session 2010: what filter + stitch does to the predictions",
        row_height=0.7,
    )
    plt.show()

### Zooming In: One Stitch and One Filter

At the scale of a whole session a gap of a few frames is smaller than a pixel, so the figure above hides what filter + stitch actually does. Here are real examples from the predictions, one frame per tick (every 5th frame is labeled).

- **Stitch:** a short gap of not-grooming inside a grooming bout is filled, so the bout is counted once instead of split in two.
- **Filter:** a predicted bout shorter than the minimum is removed.

In [ ]:
# Why: show filter + stitch at frame resolution. We find the real gaps that were stitched in
# session 2010, and the longest predicted bout that was filtered out in any session we classified.
pp_all = {}   # session id -> per-frame class after filter + stitch (from `jabs-cli postprocess`)
raw_all = {}  # session id -> per-frame raw predicted class
for pf in live_pred_files:
    sid = next((s for s in session_ids + ["2010"] if s in pf.stem), pf.stem)
    with h5py.File(str(pf), "r") as f:
        raw_all[sid] = f[f"predictions/{BEHAVIOR}/predicted_class"][0]
    cleaned = postprocess_with_cli(pf, eval_dir / f"{sid}_postprocessed.h5")
    if cleaned is not None:
        pp_all[sid] = cleaned

if "2010" in pp_all:
    raw10, pp10 = raw_all["2010"], pp_all["2010"]
    n10 = min(len(dense_labels), len(raw10))
    # Gaps of not-grooming that stitching filled (ignore ones touching the very start of the video).
    stitched = [g for g in _runs((raw10[:n10] == 0) & (pp10[:n10] == 1)) if g[0] > 25]
    stitched = sorted(sorted(stitched, key=lambda g: -g[1])[:2])
    for start, length in stitched:
        lo, hi = max(0, start - 20), min(n10 - 1, start + length + 20)
        ethogram(
            [("Labeled (a1)", dense_labels[:n10]), ("Predicted, raw", raw10[:n10]), ("Predicted, filter + stitch", pp10[:n10])],
            title=f"Stitch: session 2010, a {length}-frame gap at frame {start} was filled (gap tolerance {stitch_gap} frames)",
            xlim=(lo, hi), row_height=0.7,
        )
        plt.show()

# The longest bout the filter removed, in any session.
removed = [
    (sid, start, length)
    for sid in pp_all
    for start, length in _runs((raw_all[sid] == 1) & (pp_all[sid] == 0))
]
if removed:
    sid, start, length = max(removed, key=lambda r: r[2])
    n_s = len(raw_all[sid])
    lo, hi = max(0, start - 20), min(n_s - 1, start + length + 20)
    ethogram(
        [("Predicted, raw", raw_all[sid]), ("Predicted, filter + stitch", pp_all[sid])],
        title=f"Filter: session {sid}, a {length}-frame bout at frame {start} was removed (minimum bout {min_bout} frames)",
        xlim=(lo, hi), row_height=0.7,
    )
    plt.show()
else:
    print("No predicted bout was short enough to be filtered out in these sessions.")

In [ ]:
# Why: compare raw and filtered+stitched predictions against the dense ground truth,
# at the frame level and at the bout level (both match criteria), from the one
# evaluate run above.
if eval_json is None:
    print("No evaluation results - run the evaluation cell above first.")
else:
    frame_table = pd.concat([frame_metrics_table("raw"), frame_metrics_table("postprocessed")])
    frame_table.index = ["Raw", "Filter + stitch"]
    print("Frame level:")
    print(frame_table.round(3).to_string())

    fig, ax = plt.subplots(figsize=(7, 3.5))
    metric_bars(ax, {
        "Raw": {n: float(frame_table.loc["Raw", n]) for n in METRIC_COLORS},
        "Filter + stitch": {n: float(frame_table.loc["Filter + stitch", n]) for n in METRIC_COLORS},
    }, title="Frame-level metrics: raw vs. filter + stitch")
    plt.tight_layout()
    plt.show()

    for criterion in eval_json["criteria"]:
        rows = {}
        for stage, label in (("raw", "Raw"), ("postprocessed", "Filter + stitch")):
            b = eval_json["stages"][stage]["overall"]["bouts"][criterion]
            rows[label] = {
                "Labeled bouts": b["truth_bouts"],
                "Detected": b["detected_truth_bouts"],
                "Predicted bouts": b["predicted_bouts"],
                "Detection rate": round(b["detection_rate"] or 0, 3),
                "Precision": round(b["precision"] or 0, 3),
                "F1": round(b["f1"] or 0, 3),
                "Fragmented": b["fragmented_truth_bouts"],
                "Merged": b["merged_predicted_bouts"],
            }
        print(f"\nBout level, match criterion: {criterion}")
        print(pd.DataFrame.from_dict(rows, orient="index").to_string())

## Performance: Inter-Annotator Agreement

A classifier is only as consistent as the labels it learned from, and the dense ground truth is itself one person's judgment. **Inter-annotator agreement** asks how well two annotators agree on the same video. It is the ceiling for a classifier: asking for more agreement with a label than two people have with each other is not meaningful.

For session 2010 we have two annotation passes by the instructor, made at two different times. **This is a conceptual demonstration, not best practice.** The same person labeling twice understates real disagreement between different people; in a real project, use independent annotators.

We compare the two passes frame by frame, and then bout by bout. (JABS has no command for comparing two sets of labels, so these numbers are computed in the notebook.)

In [ ]:
# Why: measure how much two annotation passes of the same video agree, frame by frame
# and bout by bout. JABS has no command that compares two sets of labels (`jabs-cli
# evaluate` scores a trained classifier), so this uses plain NumPy.
def read_frame_labels(ann_path, n_frames, behavior):
    """Read one behavior's labels from a JABS annotation file as a per-frame array.

    Args:
        ann_path: Path to the annotation JSON.
        n_frames: Number of frames in the video.
        behavior: Behavior name to read.

    Returns:
        An int8 array: -1 unreviewed, 0 not behavior, 1 behavior.
    """
    labels = np.full(n_frames, -1, dtype=np.int8)
    d = json.loads(ann_path.read_text(encoding="utf-8"))
    for behaviors in d.get("unfragmented_labels", {}).values():
        for name, intervals in behaviors.items():
            if name.lower() != behavior:
                continue
            for iv in intervals:
                labels[int(iv["start"]):int(iv["end"])] = 1 if iv.get("present", False) else 0
    return labels


def best_iou(bout, others):
    """Find the highest intersection-over-union between one bout and a list of bouts.

    Args:
        bout: A (start, length) pair.
        others: A list of (start, length) pairs.

    Returns:
        The best IoU, or 0.0 if nothing overlaps.
    """
    s1, e1 = bout[0], bout[0] + bout[1]
    best = 0.0
    for s2, length2 in others:
        inter = max(0, min(e1, s2 + length2) - max(s1, s2))
        union = (e1 - s1) + length2 - inter
        best = max(best, inter / union if union else 0.0)
    return best


second_pass_path = project_root / "data" / "dense_gt_a2" / "jabs" / "annotations" / "2010.json"
if not second_pass_path.exists():
    print(f"[MISSING] Second annotation pass not found: {second_pass_path}")
else:
    pass1 = dense_labels
    pass2 = read_frame_labels(second_pass_path, n_dense_frames, BEHAVIOR)

    # The two passes as an ethogram, with the frames where they disagree marked below.
    both = (pass1 >= 0) & (pass2 >= 0)
    ethogram(
        [("a1 (pass 1)", pass1), ("a2 (pass 2)", pass2), ("Disagree", ((pass1 != pass2) & both).astype(int), "disagree")],
        title="Session 2010: the two annotation passes",
    )
    plt.show()

    # Dates the two passes were labeled (YYYY-MM-DD); the gap is printed when both are set.
    pass1_labeled_on = ""
    pass2_labeled_on = "2026-10-07"
    if pass1_labeled_on and pass2_labeled_on:
        from datetime import date

        gap = (date.fromisoformat(pass2_labeled_on) - date.fromisoformat(pass1_labeled_on)).days
        print(f"The two passes were labeled {gap} days apart.\n")

    # Frame level, with pass 1 as the reference (same definitions as the table above).
    a, b = pass1[both], pass2[both]
    tp, fp = int(((a == 1) & (b == 1)).sum()), int(((a == 0) & (b == 1)).sum())
    fn, tn = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 0)).sum())
    agreement = pd.DataFrame([{
        "Frames": tp + fp + fn + tn,
        "Accuracy": (tp + tn) / (tp + fp + fn + tn),
        "Recall": tp / (tp + fn) if tp + fn else 0.0,
        "Precision": tp / (tp + fp) if tp + fp else 0.0,
        "F1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
    }], index=["a2 vs a1"])
    print("Frame level (a1 as reference):")
    print(agreement.round(3).to_string())

    # Bout level: a bout counts as found if some bout in the other pass overlaps it with IoU >= 0.5.
    bouts1, bouts2 = _runs(pass1 == 1), _runs(pass2 == 1)
    found = sum(best_iou(x, bouts2) >= 0.5 for x in bouts1)
    agreed = sum(best_iou(x, bouts1) >= 0.5 for x in bouts2)
    print(f"\nBout level (IoU >= 0.5): a1 has {len(bouts1)} bouts, a2 has {len(bouts2)}")
    print(f"  a1 bouts found in a2: {found} of {len(bouts1)}")
    print(f"  a2 bouts found in a1: {agreed} of {len(bouts2)}")

### Compare With a Neighbor

Turn to the person next to you. You labeled different frames, so your classifiers learned from different examples. Look for where they agree (the same frames called grooming) as well as where they differ. Compare the classifier performance you each got on session 2010 (accuracy, F1, bouts detected after filter + stitch). Your numbers differ because your labels differ. How does the difference between the two of you compare with the agreement between the two instructor passes above?

## Performance: Biological Validation

A classifier can score well and still be wrong in a way that matters. A separate check is whether the **predicted bouts look like the behavior biologically**. Grooming is a low-activity behavior and often happens in particular places in the arena, so we check two things on the classifier's filtered and stitched predictions:

1. Does the body **move less** during predicted grooming than at other times?
2. **Where in the arena** does predicted grooming happen?

These use the sessions we classified live, plus session 2010.

In [ ]:
# Why: build the filtered+stitched predicted bouts for every session we classified
# live (every practice session plus session 2010), using the same `jabs-cli postprocess`
# step as above, so the biological checks below use the same bout-level predictions.
pp_preds = {}      # session id -> per-frame class after filter + stitch (0/1)
bio_pose = {}      # session id -> pose file path
for pf in live_pred_files:
    sid = next((s for s in session_ids + ["2010"] if s in pf.stem), pf.stem)
    pose = dense_pose_path if sid == "2010" else data_dir / f"{sid}_pose_est_v6.h5"
    if not pose.exists():
        continue
    cleaned = pp_all.get(sid)
    if cleaned is None:
        cleaned = postprocess_with_cli(pf, eval_dir / f"{sid}_postprocessed.h5")
    if cleaned is not None:
        pp_preds[sid] = cleaned
        bio_pose[sid] = pose

pred_bouts = [
    {"session": sid, "start": int(start), "end": int(start + length), "length": int(length)}
    for sid, arr in pp_preds.items()
    for start, length in _runs(arr == 1)
]
pred_bouts_df = pd.DataFrame(pred_bouts, columns=["session", "start", "end", "length"])
print(f"{len(pp_preds)} sessions, {len(pred_bouts_df)} predicted grooming bouts after filter + stitch")

In [ ]:
# Why: when does predicted grooming happen in each session (after filter + stitch),
# and how long do its bouts last? One ethogram row per session.
if pred_bouts_df.empty:
    print("No predicted grooming bouts - train a classifier first (see the cells above).")
else:
    ethogram(
        [(sid, pp_preds[sid]) for sid in sorted(pp_preds)],
        title="Predicted grooming after filter + stitch, one row per session",
    )
    plt.show()

In [ ]:
# Why: an independent check that predicted bouts capture real behavior - grooming
# should show less body movement than other frames.
CENTER_SPINE = KEYPOINT_NAMES.index("center_spine") if "center_spine" in KEYPOINT_NAMES else 6

groom_speeds, other_speeds = [], []
sessions_used = 0

for sid, pose_path in bio_pose.items():
    lab = pp_preds.get(sid)
    # Skip sessions with too few grooming frames to compare.
    if lab is None or (lab == 1).sum() < 10:
        continue
    # Center-spine position per frame, as floats so differences are exact.
    with h5py.File(str(pose_path), "r") as f:
        pts = f["poseest/points"][:, 0, CENTER_SPINE, :].astype(float)
    # Frame-to-frame displacement in pixels.
    disp = np.linalg.norm(pts[1:] - pts[:-1], axis=1)
    # Displacement has one fewer value than frames; align the labels to it.
    lab_s = lab[1:]
    groom_speeds.extend(disp[lab_s == 1].tolist())
    other_speeds.extend(disp[lab_s == 0].tolist())
    sessions_used += 1

if not groom_speeds:
    print("Not enough grooming frames to compare.")
else:
    print(f"Sessions used: {sessions_used}")
    print(f"Grooming  frames: {len(groom_speeds):6d}  mean disp: {np.mean(groom_speeds):.2f} px/frame")
    print(f"Other     frames: {len(other_speeds):6d}  mean disp: {np.mean(other_speeds):.2f} px/frame")

**Where does grooming happen?** Mice often hug the arena walls, so we check whether predicted grooming clusters near them. Each pose file stores the four **arena corners** (the same ones JABS uses for its own wall-distance features). From them we measure how far the mouse's center-spine is from the **nearest wall**, in cm, and we place every frame on a common arena map centered on the arena itself (not on the mouse).

A mouse often stays in one place for a whole session, so frames from the same session are not independent. We therefore report the distance **per session**: the session is the unit, and the question is how many sessions show grooming closer to a wall than not-grooming.

In [ ]:
# Why: check whether predicted grooming happens near the arena walls, measured against the real
# arena (the corners stored in each pose file). Results are reported per session, because the
# frames of one session are not independent of each other.
import cv2

wall_zone_cm = 5   # a frame is "at the wall" if the mouse is within this many cm of it (arena is about 52 cm wide)
min_frames = 10    # a session needs at least this many frames of each class to be compared

per_session = {}   # session id -> dict with wall distances and arena positions for each class
arena_outline_cm = None

for sid, pose_path in bio_pose.items():
    lab = pp_preds.get(sid)
    if lab is None:
        continue
    with h5py.File(str(pose_path), "r") as f:
        corners = f["static_objects/corners"][:].astype(np.float32)   # (4, 2), stored [x, y]
        cm_per_pixel = float(f["poseest"].attrs["cm_per_pixel"])
        pts = f["poseest/points"][:, 0, CENTER_SPINE, :].astype(np.float32)   # (n, 2), stored [y, x]
    xy = pts[:, ::-1]                       # to [x, y], the same order as the corners
    hull = cv2.convexHull(corners)          # the four corners in boundary order
    # Signed distance from each frame's position to the nearest wall; positive = inside the arena.
    wall_cm = np.array([cv2.pointPolygonTest(hull, (float(x), float(y)), True) for x, y in xy]) * cm_per_pixel
    centered_cm = (xy - corners.mean(axis=0)) * cm_per_pixel   # cm from the arena center
    n = min(len(lab), len(xy))
    groom, other = lab[:n] == 1, lab[:n] == 0
    if groom.sum() < min_frames or other.sum() < min_frames:
        continue
    per_session[sid] = {
        "groom_wall": wall_cm[:n][groom], "other_wall": wall_cm[:n][other],
        "groom_xy": centered_cm[:n][groom], "other_xy": centered_cm[:n][other],
    }
    if arena_outline_cm is None:
        arena_outline_cm = (hull[:, 0, :] - corners.mean(axis=0)) * cm_per_pixel

if not per_session:
    print("Not enough predicted grooming and not-grooming frames to compare.")
else:
    rows = []
    for sid, d in sorted(per_session.items()):
        rows.append({
            "Session": sid,
            "Grooming frames": len(d["groom_wall"]),
            "Grooming: median cm from wall": np.median(d["groom_wall"]),
            f"Grooming: % within {wall_zone_cm} cm": 100 * (d["groom_wall"] < wall_zone_cm).mean(),
            "Other frames": len(d["other_wall"]),
            "Other: median cm from wall": np.median(d["other_wall"]),
            f"Other: % within {wall_zone_cm} cm": 100 * (d["other_wall"] < wall_zone_cm).mean(),
        })
    session_table = pd.DataFrame(rows).set_index("Session")
    print(session_table.round(1).to_string())
    closer = int((session_table["Grooming: median cm from wall"] < session_table["Other: median cm from wall"]).sum())
    print(f"\nGrooming is closer to a wall than not-grooming (by median) in {closer} of {len(session_table)} sessions.")

    fig, axes = plt.subplots(1, 2, figsize=(18, max(8, 0.9 * len(per_session) + 3)), gridspec_kw={"width_ratios": [1.1, 1]})

    # Left: one boxplot pair per session (not grooming in blue, grooming in orange). The box spans the 25th to
    # 75th percentile with the median marked; points beyond 1.5 x the box height are drawn as outliers.
    ax = axes[0]
    sessions_sorted = sorted(per_session)
    # Matplotlib 3.10 renamed boxplot's vert=False to orientation="horizontal" (and 3.11 warns about the old name).
    import matplotlib

    horizontal = ({"orientation": "horizontal"} if tuple(int(x) for x in matplotlib.__version__.split(".")[:2]) >= (3, 10)
                  else {"vert": False})
    for i, sid in enumerate(sessions_sorted):
        for offset, key, color in ((-0.19, "other_wall", NOT_BEHAVIOR_COLOR), (0.19, "groom_wall", BEHAVIOR_COLOR)):
            ax.boxplot(
                per_session[sid][key], positions=[i + offset], widths=0.32, **horizontal, patch_artist=True, manage_ticks=False,
                boxprops=dict(facecolor=color, edgecolor="#333333", alpha=0.85),
                medianprops=dict(color="#111111", linewidth=1.5),
                whiskerprops=dict(color="#333333"), capprops=dict(color="#333333"),
                flierprops=dict(marker="o", markersize=2.5, markerfacecolor=color, markeredgecolor="none", alpha=0.5),
            )
    ax.set_yticks(range(len(sessions_sorted)))
    ax.set_yticklabels([f"{sid}  (n={len(per_session[sid]['groom_wall'])} grooming)" for sid in sessions_sorted], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Distance to the nearest wall (cm)")
    ax.set_title("Per session: grooming (orange) vs. not grooming (blue)")
    ax.legend(handles=[Patch(color=BEHAVIOR_COLOR, label="Grooming"), Patch(color=NOT_BEHAVIOR_COLOR, label="Not grooming")],
              loc="lower right")

    # Right: where the centroids are on one common arena. Many frames sit on the same spot, so dots
    # alone would pile up: bin the arena (2 cm bins) and, for each class, measure the share of that
    # class's time spent in each bin. Bin opacity is proportional to that share.
    from matplotlib.colors import to_rgb

    groom_xy = np.concatenate([d["groom_xy"] for d in per_session.values()])
    other_xy = np.concatenate([d["other_xy"] for d in per_session.values()])
    half = float(np.ceil(np.abs(arena_outline_cm).max()))
    edges = np.arange(-half, half + 2, 2.0)

    def share_layer(points, color):
        counts, _, _ = np.histogram2d(points[:, 0], points[:, 1], bins=[edges, edges])   # [x bin, y bin]
        share = counts / counts.sum()                                                    # fraction of this class's time
        layer = np.zeros(share.T.shape + (4,))
        layer[..., :3] = to_rgb(color)
        layer[..., 3] = share.T / share.max()                                            # linear: opacity is proportional to the share
        return layer

    ax = axes[1]
    extent = [-half, half, -half, half]
    ax.imshow(share_layer(other_xy, NOT_BEHAVIOR_COLOR), origin="lower", extent=extent)
    ax.imshow(share_layer(groom_xy, BEHAVIOR_COLOR), origin="lower", extent=extent)
    # The individual centroid points (center-spine positions) sit on top of the bins, with faint bin lines.
    for edge in edges:
        ax.axvline(edge, color="#BBBBBB", linewidth=0.3, zorder=0)
        ax.axhline(edge, color="#BBBBBB", linewidth=0.3, zorder=0)
    ax.scatter(other_xy[:, 0], other_xy[:, 1], s=3, color=NOT_BEHAVIOR_COLOR, alpha=0.6, linewidths=0)
    ax.scatter(groom_xy[:, 0], groom_xy[:, 1], s=3, color=BEHAVIOR_COLOR, alpha=0.6, linewidths=0)
    closed = np.vstack([arena_outline_cm, arena_outline_cm[:1]])
    ax.plot(closed[:, 0], closed[:, 1], color="#333333", linewidth=1.5)
    ax.set_aspect("equal")
    ax.invert_yaxis()  # image coordinates: y=0 is the top of the frame
    ax.set_xlabel("x, from the arena center (cm)")
    ax.set_ylabel("y, from the arena center (cm)")
    ax.set_title("Centroids in 2 cm bins (opacity proportional to share of time)")
    plt.tight_layout()
    plt.show()

## Check In

In your own words: which of the checks in this section (frame level, bout level after filter + stitch, inter-annotator agreement, biological validation) would you trust most to decide whether to use this classifier on new data, and why is it not enough on its own?

## Applying the Classifier to New Data

Once you trust a classifier, the next step is to apply it to new data from the command line. It follows the active learning loop in the GUI: you have built a classifier, and now you run it on sessions nobody has labeled. We introduce this step but do not run it here; the JABS documentation covers it in full:

- [JABS documentation](https://kumarlabjax.github.io/JABS-behavior-classifier/): user guide, CLI reference, file formats
- `jabs-classify classify` runs a trained classifier on pose files, and `jabs-cli postprocess` applies filter + stitch to the predictions.
- Validate on your own ground truth before relying on someone else's classifier, as we did here.
- [Course repository](https://github.com/KumarLabJax/getting-started-with-jabs): the notebooks, scripts and data used in this course.

## Further Reading

- Anderson & Perona (2014) - *Toward a Science of Computational Ethology*, Neuron 84(1):18-31.
- Goodwin, Nilsson et al. (2024) - *SimBA as a platform for explainable machine learning in behavioral neuroscience*, Nature Neuroscience.
- [JABS documentation](https://kumarlabjax.github.io/JABS-behavior-classifier/) - user guide, CLI reference, file formats.
- [JABS source](https://github.com/KumarLabJax/JABS-behavior-classifier)
- Kabra, Robie, Rivera-Alba, Branson & Branson (2013) - [*JAABA: interactive machine learning for automatic annotation of animal behavior*](https://www.nature.com/articles/nmeth.2281), Nature Methods 10(1):64-67. The source of the window-feature idea ([code](https://github.com/kristinbranson/JAABA)).
- Segalin et al. (2021) - [*The Mouse Action Recognition System (MARS) software pipeline for automated analysis of social behaviors in mice*](https://doi.org/10.7554/eLife.63720), eLife 10:e63720.
- Choudhary et al. (2026) - [*JAX Animal Behavior System (JABS), a genetics-informed, end-to-end advanced behavioral phenotyping platform for the laboratory mouse*](https://elifesciences.org/articles/107259), eLife 107259.
- [Course repository](https://github.com/KumarLabJax/getting-started-with-jabs)